<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/02_deep_training/notebooks/sesion_02_practica_b_receta_mlflow.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/02_deep_training/notebooks/sesion_02_practica_b_receta_mlflow.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F02_deep_training%2Fnotebooks%2Fsesion_02_practica_b_receta_mlflow.ipynb)

# SI7011 · Sesión 02 · Práctica B: una receta robusta de entrenamiento con MLflow
**Juan David Martínez Vargas · Universidad EAFIT**

## Tipo de cobertura forestal · Covertype

A partir de variables cartográficas de una parcela de 30 m × 30 m en el Bosque Nacional Roosevelt (Colorado), predice cuál de **7 tipos de cobertura forestal** domina. Es uno de los conjuntos de referencia más usados para aprendizaje profundo en datos tabulares:

- **581.012 registros**, **54 variables**: 10 numéricas (elevación, pendiente, distancias a agua, vías e incendios, sombreado) + 4 indicadores de zona silvestre + 40 indicadores de tipo de suelo;
- **7 clases muy desbalanceadas**: dos clases suman cerca del 85 % de los datos.

**Objetivo.** Partimos de un MLP profundo que entrena mal y agregamos **una decisión a la vez**, como en la diapositiva *Practice B*:

1. verificaciones antes de entrenar ($\ln C$, sobreajustar un lote);
2. inicialización He;
3. Adam / AdamW;
4. BatchNorm;
5. regularización: weight decay y Dropout;
6. warmup + coseno, recorte de gradiente y early stopping;
7. **empaquetar y desplegar** la receta elegida: predecir sobre filas crudas desde el artefacto de MLflow.

Como en S01, el notebook recorre las tres etapas con sus artefactos: **datos** (contrato y ficha), **entrenamiento** (corridas en MLflow con modelo **y** preprocesamiento) y **despliegue** (predicción sobre filas crudas, verificada).

Cada entrenamiento queda registrado en **MLflow**: hiperparámetros, curvas por época, learning rate, norma del gradiente y el mejor modelo. Al final comparamos todas las corridas y evaluamos en prueba **solo** la receta elegida con validación.

**Duración:** 90 minutos. **Entorno:** Kaggle o Colab (GPU recomendada, CPU suficiente con el tamaño por defecto).

**Notación.** $z_l=W_l a_{l-1}+b_l$, $a_l=\phi(z_l)$, $L_i$ pérdida de un ejemplo, $J$ objetivo, $J_\mathcal B$ objetivo del mini-lote, $\eta$ learning rate.

## 0. Preparación · 5 min

**En Kaggle:** agrega el dataset [Forest Cover Type Dataset](https://www.kaggle.com/datasets/uciml/forest-cover-type-dataset) con **Add Input** (archivo `covtype.csv`) y activa *Internet* para instalar MLflow.
**En Colab o local:** si no encuentra `covtype.csv`, la celda de carga lo descarga con `sklearn.datasets.fetch_covtype` (fuente UCI, ~11 MB).

In [ ]:
%pip install -q mlflow

In [ ]:
import copy
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

import mlflow
from mlflow.tracking import MlflowClient

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '| MLflow', mlflow.__version__, '| device:', device)

## 1. Carga e inspección · 5 min

In [ ]:
def load_covertype():
    # Busca covtype.csv (Kaggle: uciml/forest-cover-type-dataset); si no está, usa scikit-learn.
    roots = [Path('/kaggle/input'), Path('.')]
    hits = [p for r in roots if r.exists() for p in r.glob('**/covtype.csv')]
    if hits:
        print('Leyendo', hits[0])
        return pd.read_csv(hits[0])
    from sklearn.datasets import fetch_covtype
    print('covtype.csv no encontrado: descargando con sklearn.datasets.fetch_covtype')
    return fetch_covtype(as_frame=True).frame


df = load_covertype()
FEATURES = list(df.columns[:54])
NUMERIC = FEATURES[:10]          # las 44 restantes son indicadores 0/1
X_all = df[FEATURES].to_numpy(np.float32)
y_all = df.iloc[:, -1].to_numpy().astype(np.int64) - 1   # clases 1..7 -> 0..6
C = 7
print(df.shape)
print(df[NUMERIC].describe().loc[['mean', 'std', 'min', 'max']].T.round(1))

counts = np.bincount(y_all, minlength=C)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(np.arange(1, C + 1), 100 * counts / counts.sum(), color='#287dad')
ax.set(xlabel='Cover_Type', ylabel='% de registros', title='Distribución de clases')
fig.tight_layout(); plt.show()

### Contrato de datos y ficha

El contrato fija las 54 columnas, su orden y su tipo: 10 numéricas y 44 indicadores 0/1. Se usa ahora para revisar el archivo y en la sección 11 para rechazar filas mal formadas antes de predecir. La ficha queda como artefacto de cada corrida en MLflow.

In [ ]:
import hashlib, json

CONTRACT = {'features': FEATURES, 'numeric': NUMERIC, 'binary': FEATURES[10:], 'target': 'Cover_Type',
            'classes': list(range(1, C + 1))}


def validate_rows(frame, contract=CONTRACT):
    missing = [c for c in contract['features'] if c not in frame.columns]
    if missing:
        raise ValueError(f'Faltan columnas: {missing[:5]}{"..." if len(missing) > 5 else ""}')
    X = frame[contract['features']]
    if X.isna().any().any():
        raise ValueError(f'Valores faltantes en {X.columns[X.isna().any()].tolist()[:5]}')
    if not all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns):
        raise ValueError('Todas las columnas deben ser numéricas.')
    bad = [c for c in contract['binary'] if not X[c].isin([0, 1]).all()]
    if bad:
        raise ValueError(f'Indicadores con valores distintos de 0/1: {bad[:5]}')
    return X.to_numpy(dtype=np.float32, copy=True)


src = next((p for r in [Path('/kaggle/input'), Path('.')] if r.exists() for p in r.glob('**/covtype.csv')), None)
data_card = {'fuente': str(src) if src else 'sklearn.datasets.fetch_covtype',
             'sha256': hashlib.sha256(src.read_bytes()).hexdigest()[:16] if src else None,
             'registros': len(df), 'variables': len(FEATURES),
             'clases': {int(k): int(v) for k, v in zip(range(1, C + 1), np.bincount(y_all, minlength=C))}}
with open('data_card.json', 'w') as f:
    json.dump(data_card, f, indent=1)
print(json.dumps(data_card, indent=1))
X_all = validate_rows(df)

**Pregunta 1.** ¿Qué accuracy obtiene un clasificador que siempre predice la clase más frecuente? ¿Por qué reportaremos también **F1 macro**?

_Respuesta:_

## 2. TODO 1 · Partición y estandarización · 10 min

Implementa `prepare_data`:

1. Separa **20 % para prueba** y **20 % para validación**, con `stratify` y `random_state=SEED`.
2. Del 60 % restante toma como entrenamiento una muestra estratificada de tamaño `train_size` (si `train_size` es `None` o mayor que lo disponible, usa todo).
3. Ajusta un `StandardScaler` **solo con las 10 columnas numéricas del entrenamiento**; aplícalo a las tres particiones. Los 44 indicadores 0/1 no se modifican.
4. Devuelve arreglos `float32` para las entradas, `int64` para las etiquetas y el **escalador ajustado** (lo necesitaremos al desplegar).

Con `TRAIN_SIZE = 100_000` una época tarda pocos segundos en CPU. En GPU puedes usar `None` (≈ 349 mil ejemplos).

In [ ]:
TRAIN_SIZE = 100_000
N_NUM = len(NUMERIC)


def prepare_data(X, y, train_size=TRAIN_SIZE):
    # TODO 1: 60/20/20 estratificado, submuestra de entrenamiento y escalado de las 10 numéricas.
    # return X_train, X_val, X_test, y_train, y_val, y_test, scaler
    raise NotImplementedError('Completa TODO 1')

X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_data(X_all, y_all)
assert X_train.dtype == np.float32 and X_train.shape[1] == 54
assert np.allclose(X_train[:, :N_NUM].mean(0), 0, atol=1e-3), 'Escala solo con entrenamiento.'
assert set(np.unique(X_train[:, N_NUM:])) <= {0.0, 1.0}, 'Los indicadores 0/1 no se escalan.'
assert set(y_train) == set(range(C))
print('Entrenamiento', X_train.shape, '| validación', X_val.shape, '| prueba', X_test.shape)

### Lotes en memoria

El conjunto cabe en memoria, así que lo pasamos una sola vez a `device` y recorremos lotes con una permutación de índices. Es lo mismo que `DataLoader(..., shuffle=True)` sobre un `TensorDataset`, pero sin el costo de indexar ejemplo por ejemplo.

In [ ]:
T = {name: torch.as_tensor(arr, device=device) for name, arr in
     dict(X_train=X_train, X_val=X_val, X_test=X_test, y_train=y_train, y_val=y_val, y_test=y_test).items()}


def batches(X, y, batch_size, shuffle, generator=None):
    n = len(y)
    idx = torch.randperm(n, generator=generator).to(X.device) if shuffle else torch.arange(n, device=X.device)
    for start in range(0, n, batch_size):
        j = idx[start:start + batch_size]
        yield X[j], y[j]


xb, yb = next(batches(T['X_train'], T['y_train'], 512, shuffle=True))
print(xb.shape, xb.dtype, yb.shape, yb.dtype)

## 3. Un MLP configurable · TODO 2 y 3 · 15 min

Toda la receta vive en un diccionario `cfg`. Así cada corrida queda descrita por sus hiperparámetros y MLflow puede registrarlos.

**TODO 2 · `init_weights`.** Para cada `nn.Linear`:
- `'pytorch'`: no hagas nada (inicialización por defecto de `nn.Linear`);
- `'xavier'`: `nn.init.xavier_normal_`;
- `'he'`: `nn.init.kaiming_normal_(..., nonlinearity='relu')`, que usa $\operatorname{Var}(W)=2/n_\text{in}$;
- en `'xavier'` y `'he'`, sesgos en cero.

**TODO 3 · `make_mlp`.** `depth` bloques ocultos de ancho `width` y una capa de salida con `C` logits. Cada bloque:

$$
\texttt{Linear}\ \to\ [\texttt{BatchNorm1d}]\ \to\ \phi\ \to\ [\texttt{Dropout}]
$$

BatchNorm solo si `cfg['norm'] == 'batchnorm'`; Dropout solo si `cfg['dropout'] > 0`. Con BatchNorm el sesgo de `Linear` es redundante (lo absorbe $\beta$): usa `bias=False`.

In [ ]:
ACTS = {'relu': nn.ReLU, 'tanh': nn.Tanh, 'gelu': nn.GELU, 'leaky_relu': nn.LeakyReLU}


@torch.no_grad()
def init_weights(model, scheme):
    # TODO 2: inicialización 'pytorch', 'xavier' o 'he'.
    raise NotImplementedError('Completa TODO 2')


def make_mlp(cfg, n_in=54, n_out=C):
    # TODO 3: bloques Linear -> [BatchNorm1d] -> activación -> [Dropout], y una salida lineal.
    # Al final: init_weights(model, cfg['init']) y return model
    raise NotImplementedError('Completa TODO 3')

BASE = dict(
    depth=8, width=256, act='relu', init='pytorch', norm='none', dropout=0.0,   # arquitectura
    optimizer='sgd', lr=0.05, momentum=0.0, weight_decay=0.0,                 # optimizador
    schedule='constant', warmup_epochs=0, clip_norm=None,                     # dinámica
    epochs=20, batch_size=512, patience=None, seed=SEED,                      # entrenamiento
)

_bn = make_mlp({**BASE, 'norm': 'batchnorm', 'dropout': 0.1, 'init': 'he'})
assert any(isinstance(m, nn.BatchNorm1d) for m in _bn) and any(isinstance(m, nn.Dropout) for m in _bn)
assert _bn[0].bias is None and _bn(xb[:4].cpu()).shape == (4, C)
_he = make_mlp({**BASE, 'init': 'he'})
w = _he[2].weight
assert abs(w.var().item() * w.shape[1] - 2) < 0.2, 'He: n_in Var(W) debería ser cerca de 2.'
assert torch.all(_he[0].bias == 0)
print(make_mlp(BASE))

## 4. Verificaciones antes de entrenar · 10 min

Dos pruebas baratas que detectan la mayoría de errores (diapositiva *Sanity checks*):

1. **Pérdida inicial.** Con predicciones casi uniformes sobre $C=7$ clases, la entropía cruzada empieza cerca de $\ln 7\approx 1.95$.
2. **Sobreajustar un lote.** Un modelo y un ciclo correctos llevan la pérdida de un lote pequeño casi a 0.

Además medimos lo mismo que en la Práctica A: la norma del gradiente de la primera y la última capa oculta.

In [ ]:
def first_last_grad(model, X, y):
    model.zero_grad()
    loss = F.cross_entropy(model(X), y)
    loss.backward()
    lin = [m for m in model if isinstance(m, nn.Linear)][:-1]
    return loss.item(), lin[0].weight.grad.norm().item(), lin[-1].weight.grad.norm().item()


def overfit_one_batch(cfg, n=64, steps=300):
    set_seed(cfg['seed'])
    model = make_mlp(cfg).to(device)
    X, y = T['X_train'][:n], T['y_train'][:n]
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for _ in range(steps):
        loss = F.cross_entropy(model(X), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()


print(f'ln C = {math.log(C):.3f}')
for init in ['pytorch', 'xavier', 'he']:
    set_seed(SEED)
    m = make_mlp({**BASE, 'depth': 16, 'init': init}).to(device)
    l0, g1, gD = first_last_grad(m, T['X_train'][:2048], T['y_train'][:2048])
    print(f'{init:8s} D=16  pérdida inicial {l0:.3f} | ||grad W_1|| {g1:.2e} | ||grad W_D|| {gD:.2e} | '
          f'pérdida tras sobreajustar 64 ejemplos: {overfit_one_batch({**BASE, "depth": 16, "init": init}):.4f}')

**Pregunta 2.** ¿Las tres inicializaciones pasan la prueba de $\ln C$? ¿Cuál tiene una razón $\lVert\nabla_{W_1}J\rVert/\lVert\nabla_{W_D}J\rVert$ más lejana de 1 y qué predice eso sobre el entrenamiento de las primeras capas? ¿Qué error de código detectaría la prueba de sobreajustar un lote que la de $\ln C$ no detecta?

_Respuesta:_

## 5. Optimizador y learning rate · TODO 4 y 5 · 10 min

**TODO 4 · `make_optimizer`.**
- `'sgd'`: `torch.optim.SGD` con `momentum=cfg['momentum']` y `weight_decay=cfg['weight_decay']`;
- `'adam'`: `torch.optim.Adam` (su `weight_decay` es la penalización L2 acoplada al gradiente);
- `'adamw'`: `torch.optim.AdamW` (weight decay **desacoplado**, diapositiva *AdamW*).

**TODO 5 · `lr_factor`.** Multiplicador del learning rate en el paso `step` (empieza en 0), con `total` pasos y `warmup` pasos de calentamiento:

$$
\eta_t=\eta_0\cdot
\begin{cases}
\dfrac{t+1}{T_w}, & t<T_w\\[8pt]
1, & \texttt{constant}\\[4pt]
\tfrac12\Big(1+\cos\dfrac{\pi\,(t-T_w)}{T-T_w}\Big), & \texttt{cosine}
\end{cases}
$$

Se usa con `torch.optim.lr_scheduler.LambdaLR` y se actualiza **en cada paso**, no en cada época.

In [ ]:
def make_optimizer(model, cfg):
    # TODO 4: SGD (con momentum), Adam o AdamW según cfg['optimizer'].
    raise NotImplementedError('Completa TODO 4')


def lr_factor(step, total, warmup, schedule):
    # TODO 5: warmup lineal y luego constante o coseno.
    raise NotImplementedError('Completa TODO 5')

assert isinstance(make_optimizer(nn.Linear(2, 2), {**BASE, 'optimizer': 'adamw', 'lr': 1e-3, 'weight_decay': 0.01}),
                  torch.optim.AdamW)
f = [lr_factor(t, 100, 10, 'cosine') for t in range(100)]
assert abs(f[0] - 0.1) < 1e-9 and abs(f[9] - 1) < 1e-9 and f[-1] < 0.01 and all(a >= b for a, b in zip(f[10:], f[11:]))
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(f, label='warmup 10 + coseno'); ax.plot([lr_factor(t, 100, 0, 'constant') for t in range(100)], label='constante')
ax.set(xlabel='paso $t$', ylabel=r'$\eta_t/\eta_0$'); ax.legend(); fig.tight_layout(); plt.show()

## 6. Una época de entrenamiento · TODO 6 · 10 min

Completa `train_one_epoch`. Para cada lote: limpiar gradientes, calcular logits y pérdida, `backward`, **medir y recortar** la norma global del gradiente, `step` del optimizador y `step` del scheduler.

`torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)` devuelve la norma **antes** de recortar. Si `cfg['clip_norm']` es `None`, llama igual con `max_norm=float('inf')`: no recorta, pero mide.

Devuelve un diccionario con la pérdida media ponderada por ejemplos (`train_loss`), la norma media del gradiente (`grad_norm`) y el learning rate del último paso (`lr`).

In [ ]:
def train_one_epoch(model, optimizer, scheduler, cfg, generator):
    model.train()
    loss_sum, n, norms = 0.0, 0, []
    for X, y in batches(T['X_train'], T['y_train'], cfg['batch_size'], shuffle=True, generator=generator):
        # TODO 6: zero_grad, forward, pérdida, backward, clip_grad_norm_, optimizer.step(), scheduler.step()
        raise NotImplementedError('Completa TODO 6')
    return {'train_loss': loss_sum / n, 'grad_norm': float(np.mean(norms)),
            'lr': optimizer.param_groups[0]['lr']}

In [ ]:
@torch.no_grad()
def evaluate(model, X, y, batch_size=8192):
    model.eval()
    loss_sum, preds = 0.0, []
    for xb_, yb_ in batches(X, y, batch_size, shuffle=False):
        logits = model(xb_)
        loss_sum += F.cross_entropy(logits, yb_, reduction='sum').item()
        preds.append(logits.argmax(1))
    pred = torch.cat(preds).cpu().numpy()
    true = y.cpu().numpy()
    return {'loss': loss_sum / len(true), 'acc': accuracy_score(true, pred),
            'macro_f1': f1_score(true, pred, average='macro', zero_division=0)}, pred

## 7. MLflow: registrar cada corrida

[MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/) organiza el trabajo en **experimentos** que contienen **corridas** (*runs*). Cada corrida guarda:

| qué | función | aquí |
|---|---|---|
| hiperparámetros | `mlflow.log_params` | todo el diccionario `cfg` |
| métricas con paso | `mlflow.log_metrics(..., step=epoch)` | pérdidas, accuracy, F1 macro, $\eta$, norma del gradiente |
| resumen | `mlflow.log_metrics` | mejor época, mejor pérdida y F1 de validación, pérdida inicial |
| artefactos | `mlflow.log_artifact` | `best.pt`: configuración, pesos, **escalador y contrato de datos**; `data_card.json` |

Usamos un almacenamiento local en el directorio del notebook: una base SQLite (`mlflow.db`) y los artefactos en `mlruns/`. No hace falta un servidor.

La función `run_experiment` ya está lista: une todo lo anterior, aplica **early stopping** sobre la pérdida de validación (si `cfg['patience']` no es `None`) y conserva el mejor estado. **Léela antes de seguir.**

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
EXPERIMENT = 'S02-covertype-receta'
mlflow.set_experiment(EXPERIMENT)


def run_experiment(cfg, run_name, verbose=True):
    cfg = {**BASE, **cfg}
    set_seed(cfg['seed'])
    model = make_mlp(cfg).to(device)
    optimizer = make_optimizer(model, cfg)
    steps_per_epoch = math.ceil(len(T['y_train']) / cfg['batch_size'])
    total, warmup = cfg['epochs'] * steps_per_epoch, cfg['warmup_epochs'] * steps_per_epoch
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda t: lr_factor(t, total, warmup, cfg['schedule']))
    gen = torch.Generator().manual_seed(cfg['seed'])

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(cfg)
        mlflow.log_param('n_params', sum(p.numel() for p in model.parameters()))
        l0, g1, gD = first_last_grad(model, T['X_train'][:2048], T['y_train'][:2048])
        mlflow.log_metrics({'init_loss': l0, 'init_grad_W1': g1, 'init_grad_WD': gD})

        best, best_state, bad = {'val_loss': float('inf')}, None, 0
        for epoch in range(cfg['epochs']):
            t0 = time.time()
            tr = train_one_epoch(model, optimizer, scheduler, cfg, gen)
            va, _ = evaluate(model, T['X_val'], T['y_val'])
            row = {**tr, 'val_loss': va['loss'], 'val_acc': va['acc'], 'val_macro_f1': va['macro_f1'],
                   'epoch_time': time.time() - t0}
            if not all(map(math.isfinite, row.values())):
                mlflow.set_tag('status', 'diverged')
                print(f'  época {epoch}: valores no finitos, se detiene la corrida')
                break
            mlflow.log_metrics(row, step=epoch)
            if row['val_loss'] < best['val_loss']:
                best, best_state, bad = {**row, 'epoch': epoch}, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
            if verbose:
                print(f"  época {epoch:2d} | train {row['train_loss']:.4f} | val {row['val_loss']:.4f} | "
                      f"F1m {row['val_macro_f1']:.3f} | lr {row['lr']:.2e} | ||g|| {row['grad_norm']:.2f}")
            if cfg['patience'] is not None and bad >= cfg['patience']:
                mlflow.set_tag('early_stopped', f'epoch {epoch}')
                break

        if best_state is not None:
            mlflow.log_metrics({'best_epoch': best['epoch'], 'best_val_loss': best['val_loss'],
                                'best_val_acc': best['val_acc'], 'best_val_macro_f1': best['val_macro_f1']})
            torch.save({'cfg': cfg, 'state_dict': best_state, 'contract': CONTRACT,
                        'mean': torch.tensor(scaler.mean_, dtype=torch.float32),
                        'scale': torch.tensor(scaler.scale_, dtype=torch.float32)}, 'best.pt')
            mlflow.log_artifact('best.pt')
            mlflow.log_artifact('data_card.json')
    return best

## 8. La escalera de experimentos · TODO 7 · 25 min

Cada corrida cambia **una** decisión respecto a la anterior. Las dos primeras están escritas; completa las demás siguiendo la diapositiva *Practice B*. Después de cada corrida, mira las curvas (sección 9) y anota una observación.

| corrida | cambio respecto a la anterior |
|---|---|
| `R0-baseline` | MLP de 8 capas, init por defecto, SGD sin momentum, $\eta=0.05$ |
| `R1-he` | `init='he'` |
| `R2-adamw` | `optimizer='adamw'`, `lr=1e-3` (todavía `weight_decay=0`) |
| `R3-batchnorm` | `norm='batchnorm'` |
| `R4-regularizacion` | `weight_decay=1e-2` y `dropout=0.1` |
| `R5-receta` | `schedule='cosine'`, `warmup_epochs=1`, `clip_norm=1.0`, `patience=5`, `epochs=30` |

Con `TRAIN_SIZE = 100_000` cada corrida tarda 1–2 minutos en CPU y unos segundos en GPU.

In [ ]:
R = {}
R['R0-baseline'] = dict()                         # BASE tal cual
R['R1-he'] = {**R['R0-baseline'], 'init': 'he'}

In [ ]:
# TODO 7: define R2-adamw, R3-batchnorm, R4-regularizacion y R5-receta, cada una a partir de la anterior.

for name, cfg in R.items():
    print(f'=== {name}: { {k: v for k, v in cfg.items()} }')
    best = run_experiment(cfg, name)
    print(f"  -> mejor época {best.get('epoch')} | val_loss {best['val_loss']:.4f} | "
          f"F1 macro {best.get('val_macro_f1', float('nan')):.3f}")

## 9. Comparar corridas con MLflow · 10 min

`mlflow.search_runs` devuelve un `DataFrame` con parámetros (`params.*`), métricas finales (`metrics.*`) y etiquetas (`tags.*`) de todas las corridas del experimento. `MlflowClient().get_metric_history` devuelve la curva completa de una métrica.

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=['attributes.start_time ASC'])
runs = runs.drop_duplicates('tags.mlflow.runName', keep='last').set_index('tags.mlflow.runName')
cols = ['params.init', 'params.optimizer', 'params.lr', 'params.norm', 'params.dropout', 'params.weight_decay',
        'params.schedule', 'metrics.init_loss', 'metrics.best_epoch', 'metrics.best_val_loss',
        'metrics.best_val_acc', 'metrics.best_val_macro_f1']
table = runs[[c for c in cols if c in runs]].rename(columns=lambda c: c.split('.', 1)[1])
table.round(4)

In [ ]:
client = MlflowClient()


def history(run_id, key):
    h = client.get_metric_history(run_id, key)
    return pd.Series({m.step: m.value for m in h}).sort_index()


keys = [('train_loss', 'pérdida de entrenamiento'), ('val_loss', 'pérdida de validación'),
        ('val_macro_f1', 'F1 macro (validación)'), ('grad_norm', r'$\|\nabla J_\mathcal{B}\|$ medio (antes de recortar)')]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for name, run in runs.iterrows():
    for ax, (k, title) in zip(axes.flat, keys):
        h = history(run.run_id, k)
        if len(h):
            ax.plot(h.index, h.values, marker='.', label=name)
            ax.set_title(title)
for ax in axes.flat:
    ax.set_xlabel('época')
axes[1, 1].set_yscale('log')
axes[0, 0].legend(fontsize=8)
fig.tight_layout(); plt.show()

**Pregunta 3 · por corrida.** Para cada paso de la escalera, explica con las curvas qué cambió y por qué, usando los conceptos de la sesión:

- **R0 → R1:** ¿cambió la velocidad inicial? Relaciónalo con `init_grad_W1` / `init_grad_WD` y la Práctica A.
- **R1 → R2:** ¿qué aporta escalar cada parámetro por su historia de gradientes?
- **R2 → R3:** ¿qué pasa con la norma del gradiente y con la estabilidad de la curva de validación?
- **R3 → R4:** ¿se redujo la brecha $J_\text{val}-J_\text{train}$? ¿Mejoró el F1 macro o solo la pérdida?
- **R4 → R5:** ¿qué hizo el coseno al final del entrenamiento? ¿Se activó el early stopping?

_Respuestas:_

### Opcional · interfaz web de MLflow

- **Colab:** ejecuta la celda siguiente; abre la interfaz en una pestaña nueva.
- **Kaggle:** no permite abrir puertos. Descarga `mlflow.db` y la carpeta `mlruns/` (panel *Output*) y en tu computador ejecuta `mlflow ui --backend-store-uri sqlite:///mlflow.db`.
- **Local:** la misma instrucción en una terminal, en la carpeta del notebook; abre <http://127.0.0.1:5000>.

En la interfaz, selecciona varias corridas y usa **Compare** para ver parámetros y curvas lado a lado.

In [ ]:
try:
    from google.colab import output
    import subprocess
    subprocess.Popen(['mlflow', 'ui', '--backend-store-uri', 'sqlite:///mlflow.db', '--port', '5000'])
    time.sleep(5)
    output.serve_kernel_port_as_window(5000)
except ImportError:
    print('No estás en Colab: usa `mlflow ui --backend-store-uri sqlite:///mlflow.db` en una terminal.')

## 10. Evaluación final en prueba · 5 min

Elige la receta con **validación** (menor `best_val_loss`, o mayor `best_val_macro_f1` si lo justificas) y solo entonces evalúa en prueba. El modelo se recupera desde los artefactos de MLflow.

In [ ]:
CRITERION = 'metrics.best_val_loss'              # o 'metrics.best_val_macro_f1' (entonces ascending=False)
chosen = runs.sort_values(CRITERION, ascending=True).iloc[0]
print('Receta elegida:', chosen.name)

path = mlflow.artifacts.download_artifacts(run_id=chosen.run_id, artifact_path='best.pt')
ckpt = torch.load(path, map_location=device, weights_only=False)
model = make_mlp(ckpt['cfg']).to(device)
model.load_state_dict(ckpt['state_dict'])

test_metrics, pred_test = evaluate(model, T['X_test'], T['y_test'])
with mlflow.start_run(run_id=chosen.run_id):
    mlflow.log_metrics({'test_loss': test_metrics['loss'], 'test_acc': test_metrics['acc'],
                        'test_macro_f1': test_metrics['macro_f1']})
print({k: round(v, 4) for k, v in test_metrics.items()})
print(classification_report(y_test, pred_test, target_names=[f'tipo {k}' for k in range(1, C + 1)],
                            digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=range(1, C + 1),
                                        normalize='true', values_format='.2f', ax=ax, colorbar=False)
ax.set(title='Matriz de confusión normalizada por fila (prueba)', xlabel='predicha', ylabel='real')
fig.tight_layout(); plt.show()

## 11. Despliegue: de la corrida de MLflow a una predicción · TODO 8 · 15 min

El artefacto `best.pt` de la corrida elegida contiene todo lo necesario para predecir: arquitectura (`cfg`), pesos, media y escala de las 10 numéricas y el contrato de datos. Ningún objeto de este notebook (`scaler`, `T`, `model`) debe hacer falta.

**TODO 8.** Implementa `predict_raw(ckpt, frame)`:
1. validar `frame` con `validate_rows(frame, ckpt['contract'])`;
2. estandarizar **solo** las 10 numéricas con `ckpt['mean']` y `ckpt['scale']`;
3. reconstruir el modelo con `make_mlp(ckpt['cfg'])` y `ckpt['state_dict']`, en modo evaluación y sin gradientes;
4. devolver un `DataFrame` con `Cover_Type` (1–7) y `prob` (probabilidad de esa clase).

In [ ]:
@torch.no_grad()
def predict_raw(ckpt, frame):
    # TODO 8: contrato -> estandarizar numéricas -> modelo -> softmax -> Cover_Type 1..7
    raise NotImplementedError('Completa TODO 8')

### Verificaciones del despliegue

1. **Consistencia:** sobre filas crudas de prueba, `predict_raw` reproduce exactamente las predicciones de la sección 10.
2. **Contrato:** una fila mal formada se rechaza antes de llegar al modelo.
3. **Inferencia por lotes:** un archivo CSV crudo entra y sale un CSV de predicciones; medimos latencia por fila y por lote.
4. **Exportación:** el modelo exportado con `torch.export` (archivo `.pt2`) da las mismas salidas sin necesitar el código de `make_mlp`.

In [ ]:
ckpt = torch.load(mlflow.artifacts.download_artifacts(run_id=chosen.run_id, artifact_path='best.pt'),
                  map_location='cpu', weights_only=True)

# Filas crudas de prueba: se deshace la estandarización de las numéricas.
raw = X_test.copy(); raw[:, :N_NUM] = scaler.inverse_transform(raw[:, :N_NUM])
raw_test = pd.DataFrame(raw, columns=FEATURES)
raw_test[CONTRACT['binary']] = raw_test[CONTRACT['binary']].round().astype(int)

out = predict_raw(ckpt, raw_test)
assert (out['Cover_Type'].to_numpy() - 1 == pred_test).mean() > 0.999, 'El artefacto no reproduce las predicciones.'
print('1. Consistencia:', f"{(out['Cover_Type'].to_numpy() - 1 == pred_test).mean():.4%}", 'de coincidencia')

bad = raw_test.head(3).copy(); bad.loc[bad.index[0], CONTRACT['binary'][0]] = 7
try:
    predict_raw(ckpt, bad)
except ValueError as e:
    print('2. Contrato: fila rechazada ->', e)

raw_test.head(20_000).to_csv('nuevos_registros.csv', index=False)
t0 = time.perf_counter()
batch = pd.read_csv('nuevos_registros.csv')
pred = predict_raw(ckpt, batch)
pd.concat([batch, pred], axis=1).to_csv('predicciones.csv', index=False)
t_batch = time.perf_counter() - t0
t0 = time.perf_counter()
for i in range(50):
    predict_raw(ckpt, batch.iloc[[i]])
t_one = (time.perf_counter() - t0) / 50
print(f'3. Lote de {len(batch)} filas: {t_batch:.2f} s ({1e6 * t_batch / len(batch):.1f} µs por fila) | '
      f'una fila a la vez: {1e3 * t_one:.1f} ms por fila')

In [ ]:
model_cpu = make_mlp(ckpt['cfg']); model_cpu.load_state_dict(ckpt['state_dict']); model_cpu.eval()
example = torch.zeros(8, len(FEATURES))
exported = torch.export.export(model_cpu, (example,), dynamic_shapes=({0: torch.export.Dim('batch')},))
torch.export.save(exported, 'covertype_mlp.pt2')
reloaded = torch.export.load('covertype_mlp.pt2').module()
xs = torch.as_tensor(validate_rows(raw_test.head(256)))
xs[:, :N_NUM] = (xs[:, :N_NUM] - ckpt['mean']) / ckpt['scale']
with torch.no_grad():
    diff = (reloaded(xs) - model_cpu(xs)).abs().max().item()
print('4. torch.export: diferencia máxima entre logits =', diff)
with mlflow.start_run(run_id=chosen.run_id):
    mlflow.log_artifact('covertype_mlp.pt2')
    mlflow.log_metric('latency_ms_one_row', 1e3 * t_one)

**Pregunta 4 · despliegue.** (a) ¿Por qué la predicción fila por fila es mucho más lenta por ejemplo que por lotes? (b) El archivo `.pt2` contiene el modelo pero **no** el escalador ni el contrato: ¿qué tendría que acompañarlo en producción y qué pasaría si se olvida? (c) ¿Qué otra verificación agregarías al contrato para Covertype (pistas: rangos, exactamente un indicador de zona activo)?

_Respuesta:_

### Interpretación y entrega

1. ¿Qué decisión de la escalera tuvo el mayor efecto sobre `best_val_loss`? ¿Y sobre el F1 macro? ¿Coinciden?
2. ¿Qué clase tiene menor recall en prueba? Relaciónalo con su frecuencia.
3. Toma una corrida que no haya funcionado bien y diagnostícala con el **mapa de diagnóstico** de las diapositivas: síntoma, qué mediste, qué cambiarías.
4. ¿Por qué no usamos la prueba para elegir entre R0–R5?

**Entrega:** notebook con los ocho TODO, las corridas R0–R5 registradas en MLflow, la tabla y las curvas de la sección 9, la evaluación final, las verificaciones de despliegue (sección 11) y las respuestas. Incluye el archivo `mlflow.db` (y `mlruns/`) o una captura de la vista *Compare* de MLflow. No se exige un F1 mínimo: se evalúa la implementación y la interpretación.

**Extensiones (opcionales, cada una como una corrida nueva en MLflow):**
- profundidad 16 o 32 con y sin BatchNorm: ¿dónde aparece la degradación de la diapositiva *Deeper is not automatically better*?
- `act='gelu'` o `'leaky_relu'`;
- `TRAIN_SIZE=None` (todo el entrenamiento) en GPU: ¿sigue siendo útil la regularización?
- pérdida con pesos por clase (`F.cross_entropy(..., weight=...)`) para mejorar el F1 macro;
- excluir sesgos y parámetros de BatchNorm del weight decay (dos grupos de parámetros en AdamW).

## Referencias
- Blackard y Dean (1999), *Comparative accuracies of artificial neural networks and discriminant analysis in predicting forest cover types from cartographic variables*, Computers and Electronics in Agriculture, 24(3), 131–151.
- [UCI · Covertype](https://archive.ics.uci.edu/dataset/31/covertype) (CC BY 4.0) · [Kaggle · Forest Cover Type Dataset](https://www.kaggle.com/datasets/uciml/forest-cover-type-dataset) · [`sklearn.datasets.fetch_covtype`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html).
- Gorishniy et al. (2021), *Revisiting Deep Learning Models for Tabular Data*, NeurIPS (Covertype como referencia de MLP en datos tabulares).
- Loshchilov y Hutter (2019), *Decoupled Weight Decay Regularization*, ICLR · Ioffe y Szegedy (2015), *Batch Normalization*, ICML.
- [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/) · [PyTorch · `LambdaLR`](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.LambdaLR.html) · [`clip_grad_norm_`](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html).